In [1]:
import numpy as np
from random import randint
from scipy.sparse import rand
import numpy as np
from numpy.linalg import norm
from scipy.sparse import diags, csc_matrix
from scipy.sparse.linalg import bicgstab, spilu 
import time

In [2]:
shape = int(input())
if shape < 3:
        exit()

matrix = rand(shape, shape, density=0.6, random_state=randint(112, 154))


for i in matrix.toarray().round(3):
    for j in i:
        print(f'{j} ', end=" ")
    print('\n')
print('\n')
b = np.random.randint(5, 53, shape)
for i in b:
    print(f'{i} ', end=" ")
print('\n')
print(type(matrix))

0.656  0.08  0.692  0.113  0.216  0.0  0.749  0.841  0.0  0.668  

0.0  0.0  0.0  0.0  0.927  0.0  0.106  0.488  0.138  0.0  

0.0  0.0  0.01  0.903  0.119  0.0  0.472  0.335  0.714  0.049  

0.768  0.0  0.0  0.892  0.895  0.87  0.0  0.181  0.712  0.0  

0.448  0.628  0.328  0.0  0.104  0.873  0.621  0.918  0.44  0.536  

0.002  0.0  0.91  0.357  0.0  0.0  0.99  0.0  0.155  0.123  

0.692  0.928  0.0  0.0  0.354  0.434  0.0  0.58  0.0  0.0  

0.247  0.326  0.0  0.0  0.193  0.856  0.41  0.0  0.0  0.878  

0.0  0.874  0.038  0.0  0.777  0.0  0.512  0.206  0.577  0.0  

0.0  0.0  0.0  0.822  0.0  0.0  0.0  0.766  0.061  0.0  



44  42  25  7  33  12  43  22  29  6  

<class 'scipy.sparse._coo.coo_matrix'>


In [3]:
matrix = csc_matrix(matrix)

In [7]:
class BiCGMethod:
    def __init__(self, matrix, b, x0=None, eps=1e-5):
        self.matrix = matrix
        self.b = b
        self.eps = eps
        self.shape = matrix.shape[0]
        self.x0 = np.array([0] * self.shape) if x0 is None else x0
        self.k = 0
        
    def solve(self):
        r0 = self.b - self.matrix @ self.x0 # невязка
        x0 = self.x0 # начальное приближение
        r2 = r0 # выбирается вектор
        rho0 = 1
        alpha0 = 1
        omega0 = 1
        v0 = np.array([0] * self.shape)
        p0 = np.array([0] * self.shape)
        while True:
            rho = r2 @ r0
            beta = (rho * alpha0) / (rho0 * omega0)
            p = r0 + beta * (p0 - omega0 * v0)
            v = self.matrix @ p
            alpha = rho / (r2 @ v)
            s = r0 - alpha * v
            t = self.matrix @ s
            omega = (t @ s) / (t @ t)
            x = x0 + omega * s + alpha * p
            r = s - omega * t


            self.k += 1
            if norm(r) < self.eps: # норма заданной невязки
                break
            r0 = r
            rho0 = rho
            alpha0 = alpha
            omega0 = omega
            v0 = v
            p0 = p
            x0 = x
        return x
    
    def print_solution(self):
        start_timeBiCGM = time.time()
        x = self.solve()
        print("BiCGMethod time: --- %s seconds ---\n" % (time.time() - start_timeBiCGM))
        start_timeNumPy = time.time()
        x2 = np.linalg.solve(self.matrix.toarray(), self.b)
        print("NumPy time: --- %s seconds ---\n" % (time.time() - start_timeNumPy))
        # with open(self.output, 'w') as f:
        print('My solve:\n')
        print(f'{x.round(5)}\n')
        print(f'EPS = {self.eps}\n')
        print(f'Shape = {self.shape}\n')
        print(f'Count of iterations = {self.k}\n')
        # print(f'Mean = {np.mean(x)}\n') # среднее
        print('\nNumPy solve:\n')
        print(f'{x2.round(5)}\n')
        # print(f'Mean = {np.mean(x2)}\n')


In [8]:
solver = BiCGMethod(matrix, b, eps=1e-5)
solver.print_solution()

BiCGMethod time: --- 0.008104562759399414 seconds ---

NumPy time: --- 0.00514531135559082 seconds ---

My solve:

[  75.04408  -46.1016  -187.69614  -19.61134   15.11262   16.795
  219.22775   36.50329  -95.36002 -101.12381]

EPS = 1e-05

Shape = 10

Count of iterations = 13


NumPy solve:

[  75.04408  -46.1016  -187.69614  -19.61134   15.11263   16.795
  219.22775   36.50329  -95.36002 -101.12381]

